In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from scipy import stats
import statsmodels.stats.multitest as multitest
import warnings
import os

warnings.filterwarnings('ignore')

# =============================================================================
# Settings
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# Force math text to support bold-italic properly
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'
plt.rcParams['mathtext.bfit'] = 'Arial:italic:bold'

def lighten_hex(hex_color, factor=0.3):
    r, g, b = mc.to_rgb(hex_color)
    return mc.to_hex((r + (1-r)*factor, g + (1-g)*factor, b + (1-b)*factor))

color_xylitol = lighten_hex('#008000', 0.3)
color_erythritol = lighten_hex('#FF0000', 0.3)

def clean_and_convert_to_nan(vals):
    s_vals = pd.Series(vals).astype(str).str.strip()
    s_vals = s_vals.str.replace(r'\.E\+', 'E+', regex=True)
    s_vals = s_vals.str.replace(r'\.E\-', 'E-', regex=True)
    s_vals = s_vals.replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', '', 'None'], np.nan)
    return pd.to_numeric(s_vals, errors='coerce')

# =============================================================================
# Plotting Function for 1x3 Panel
# =============================================================================
def plot_1x3_scatter_qval(df_merge, x_cols, x_labels, y_col, y_label, color_hex, text_positions, out_filename):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

    # Calculate p-values for FDR correction
    p_vals = []
    valid_dfs = []
    for x_col in x_cols:
        valid_df = df_merge[[x_col, y_col]].dropna()
        valid_dfs.append(valid_df)
        if len(valid_df) >= 3:
            if valid_df[x_col].var() == 0 or valid_df[y_col].var() == 0:
                p_vals.append(1.0)
            else:
                _, p = stats.pearsonr(valid_df[x_col], valid_df[y_col])
                p_vals.append(p)
        else:
            p_vals.append(1.0)

    # FDR Correction
    _, q_vals, _, _ = multitest.multipletests(p_vals, alpha=0.05, method='fdr_bh')

    # Plotting panels
    for i, (x_col, x_label) in enumerate(zip(x_cols, x_labels)):
        ax = axes[i]
        valid_df = valid_dfs[i]

        if len(valid_df) < 3:
            ax.set_visible(False)
            continue

        if valid_df[x_col].var() == 0 or valid_df[y_col].var() == 0:
            r_val = 0.0
        else:
            r_val, _ = stats.pearsonr(valid_df[x_col], valid_df[y_col])

        q_val = q_vals[i]

        # Solid black line for strong correlation (r >= 0.4), solid gray line otherwise
        if abs(r_val) >= 0.4:
            line_style = {'linewidth': 2, 'color': 'black', 'alpha': 1.0, 'linestyle': '-'}
        else:
            line_style = {'linewidth': 2, 'color': 'gray', 'alpha': 0.5, 'linestyle': '-'}

        sns.regplot(x=x_col, y=y_col, data=valid_df, ax=ax, color=color_hex,
                    scatter_kws={'s': 60, 'alpha': 0.7, 'edgecolors': 'white', 'linewidths': 0.5},
                    line_kws=line_style)

        ax.axhline(0, color='black', linestyle=':', linewidth=1.5)
        if valid_df[x_col].min() < 0:
            ax.axvline(0, color='black', linestyle=':', linewidth=1.5)

        ax.set_ylabel(y_label if i == 0 else '', fontsize=14, fontweight='bold', labelpad=10)
        ax.set_xlabel(x_label, fontsize=14, fontweight='bold', labelpad=10)

        ax.tick_params(axis='both', labelsize=12, width=1.5, length=5)
        for spine in ['left', 'bottom']:
            ax.spines[spine].set_linewidth(1.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_box_aspect(1)

        q_text = "q < 0.001" if q_val < 0.001 else f"q = {q_val:.3f}"
        stats_text = f"$r = {r_val:.2f}$\n{q_text}"

        # Positioning logic with white semi-transparent background to prevent line overlap
        text_bbox = dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2')
        if text_positions[i] == 'top_left':
            ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                    fontsize=14, fontweight='normal', va='top', ha='left', color='black', bbox=text_bbox)
        else:
            ax.text(0.95, 0.05, stats_text, transform=ax.transAxes,
                    fontsize=14, fontweight='normal', va='bottom', ha='right', color='black', bbox=text_bbox)

    plt.tight_layout()
    plt.savefig(out_filename, dpi=DPI_SETTING, bbox_inches='tight', transparent=True)
    plt.close()

# =============================================================================
# Data Execution
# =============================================================================
df_butyrate = pd.read_csv('Butyrate(mM).csv')
donor_cols = [c for c in df_butyrate.columns if c.startswith('HS-')]
y_label_butyrate = r'$\Delta$ Butyrate (mM)'

# -----------------------------------------------------------------------------
# Figure 5l: Xylitol (16S Anaerostipes)
# -----------------------------------------------------------------------------
df_16s = pd.read_csv('(KULFFI)_16S_Genus_level.csv')
df_16s['Donor'] = df_16s['Subject'].astype(str).apply(lambda x: "-".join(x.split('-')[:2]) if x.startswith('HS-') else np.nan)
df_16s['Condition'] = df_16s['Subject'].astype(str).apply(lambda x: "-".join(x.split('-')[2:]) if x.startswith('HS-') else np.nan)

taxa_cols = [c for c in df_16s.columns if c.startswith('d__')]
row_sums = df_16s[taxa_cols].sum(axis=1)
# Normalize to relative abundance if not already
if row_sums.mean() > 10:
    df_16s[taxa_cols] = df_16s[taxa_cols].div(df_16s[taxa_cols].sum(axis=1), axis=0)

anaero_cols = [c for c in taxa_cols if 'Anaerostipes' in c]

if anaero_cols:
    anaero_col = anaero_cols[0]

    # Convert relative abundance to Percentage (%)
    ctrl_rows = df_16s[df_16s['Condition'] == 'CUL'].set_index('Donor')[anaero_col] * 100
    sub_rows_x = df_16s[df_16s['Condition'] == 'Xylitol'].set_index('Donor')[anaero_col] * 100

    ctrl_buty_x = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
    sub_buty_x = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Xylitol'][donor_cols].iloc[0])
    delta_buty_x = sub_buty_x - ctrl_buty_x
    delta_buty_x.index = [c.strip() for c in donor_cols]

    df_merge_x = pd.DataFrame({'buty_delta': delta_buty_x})
    df_merge_x['init'] = ctrl_rows
    df_merge_x['post'] = sub_rows_x
    df_merge_x['delta'] = df_merge_x['post'] - df_merge_x['init']

    # using \mathbfit for guaranteed Bold-Italic in Matplotlib
    x_labels_x = [
        r'Initial $\mathbfit{Anaerostipes}$ spp. (%)',
        r'Post-treatment $\mathbfit{Anaerostipes}$ spp. (%)',
        r'$\Delta$ $\mathbfit{Anaerostipes}$ spp. (%)'
    ]

    positions_x = ['bottom_right', 'bottom_right', 'bottom_right']

    plot_1x3_scatter_qval(df_merge_x, ['init', 'post', 'delta'], x_labels_x, 'buty_delta', y_label_butyrate, color_xylitol, positions_x, 'Figure_5l.pdf')

# -----------------------------------------------------------------------------
# Figure 5m: Erythritol (qPCR Butyricicoccus)
# -----------------------------------------------------------------------------
try:
    df_tax_ery = pd.read_csv('Butyricicoccus(qPCR).csv')
except UnicodeDecodeError:
    df_tax_ery = pd.read_csv('Butyricicoccus(qPCR).csv', encoding='shift_jis')

ctrl_tax_e = clean_and_convert_to_nan(df_tax_ery[df_tax_ery['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_tax_e = clean_and_convert_to_nan(df_tax_ery[df_tax_ery['KULFFI'].str.strip() == 'Erythritol'][donor_cols].iloc[0])

init_e = np.log10(ctrl_tax_e + 1)
post_e = np.log10(sub_tax_e + 1)
delta_e = post_e - init_e

ctrl_buty_e = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_buty_e = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Erythritol'][donor_cols].iloc[0])
delta_buty_e = sub_buty_e - ctrl_buty_e

df_merge_e = pd.DataFrame({'buty_delta': delta_buty_e.values}, index=donor_cols)
df_merge_e['init'] = init_e.values
df_merge_e['post'] = post_e.values
df_merge_e['delta'] = delta_e.values

x_labels_e = [
    r'Initial $\mathbfit{Butyricicoccus}$ spp.' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)',
    r'Post-treatment $\mathbfit{Butyricicoccus}$ spp.' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)',
    r'$\Delta$ $\mathbfit{Butyricicoccus}$ spp.' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)'
]

# Erythritol: Initial is top-left, others are bottom-right
positions_e = ['top_left', 'bottom_right', 'bottom_right']

plot_1x3_scatter_qval(df_merge_e, ['init', 'post', 'delta'], x_labels_e, 'buty_delta', y_label_butyrate, color_erythritol, positions_e, 'Figure_5m.pdf')